In [6]:
!pip install pyspark

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week5Assignment") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


In [7]:
file_path = "/content/superstore_raw..csv"
print(file_path)

/content/superstore_raw..csv


In [8]:
df = spark.read.csv(
    "/content/superstore_raw..csv",
    header=True,
    inferSchema=True
)

df.show(5, truncate=False)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+--------+--------+--------+--------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name  |Segment  |Country      |City           |State     |Postal Code|Region|Product ID     |Category       |Sub-Category|Product Name                                               |Sales   |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+--------+--------+--------+--------+
|1     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class  |CG-12520   |Claire Gute    |Consumer |Un

In [9]:
print("Column Names:")
print(df.columns)

print("\nSchema:")
df.printSchema()

Column Names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

Schema:
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string

In [10]:
# Q3: Remove duplicate rows based on Customer ID and Order Date

print("Total rows before removing duplicates:", df.count())

df_no_duplicates = df.dropDuplicates(["Customer ID", "Order Date"])

print("Total rows after removing duplicates:", df_no_duplicates.count())

df_no_duplicates.show(5, truncate=False)

Total rows before removing duplicates: 9994
Total rows after removing duplicates: 4992
+------+--------------+----------+---------+--------------+-----------+-------------+--------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+----------------------------------------------------------+--------+--------+--------+---------+
|Row ID|Order ID      |Order Date|Ship Date|Ship Mode     |Customer ID|Customer Name|Segment |Country      |City         |State     |Postal Code|Region |Product ID     |Category       |Sub-Category|Product Name                                              |Sales   |Quantity|Discount|Profit   |
+------+--------------+----------+---------+--------------+-----------+-------------+--------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+----------------------------------------------------------+--------+--------+--------+---------+
|1300  |CA-2015-121391|10/4/

In [12]:
# Q4: Filter West region and calculate average sales by category
from pyspark.sql.functions import col, avg, expr

# Safely convert Sales to numeric.
# Invalid values such as 'Black' will become NULL.
df_sales = df.withColumn(
    "Sales_Numeric",
    expr("try_cast(Sales as double)")
)
result_q4 = (
    df_sales
    .filter(
        (col("Region") == "West") &
        (col("Sales_Numeric").isNotNull())
    )
    .groupBy("Category")
    .agg(
        avg("Sales_Numeric").alias("Average_Sales")
    )
)
result_q4.show()

+---------------+------------------+
|       Category|     Average_Sales|
+---------------+------------------+
|Office Supplies|117.48907552370453|
|      Furniture|360.59540420899896|
|     Technology|422.64417449664415|
+---------------+------------------+



In [14]:
# Q6: Count records for each city and show cities with count > 100

from pyspark.sql.functions import count, col

result_q6 = (
    df.groupBy("City")
      .agg(count("*").alias("Total_Count"))
      .filter(col("Total_Count") > 100)
      .orderBy(col("Total_Count").desc())
)

result_q6.show()

+-------------+-----------+
|         City|Total_Count|
+-------------+-----------+
|New York City|        915|
|  Los Angeles|        747|
| Philadelphia|        537|
|San Francisco|        510|
|      Seattle|        428|
|      Houston|        377|
|      Chicago|        314|
|     Columbus|        222|
|    San Diego|        170|
|  Springfield|        163|
|       Dallas|        157|
| Jacksonville|        125|
|      Detroit|        115|
+-------------+-----------+



In [7]:
# Q10: Convert Order Date to timestamp and rename it to event_time

from pyspark.sql.functions import to_timestamp, col

df_q10 = (
    df.withColumn(
        "Order Date",
        to_timestamp(col("Order Date"), "M/d/yyyy")
    )
    .withColumnRenamed("Order Date", "event_time")
)

df_q10.select("event_time").show(10, truncate=False)

df_q10.select("event_time").printSchema()

+-------------------+
|event_time         |
+-------------------+
|2016-11-08 00:00:00|
|2016-11-08 00:00:00|
|2016-06-12 00:00:00|
|2015-10-11 00:00:00|
|2015-10-11 00:00:00|
|2014-06-09 00:00:00|
|2014-06-09 00:00:00|
|2014-06-09 00:00:00|
|2014-06-09 00:00:00|
|2014-06-09 00:00:00|
+-------------------+
only showing top 10 rows
root
 |-- event_time: timestamp (nullable = true)



In [9]:
# creating email and username column
from pyspark.sql.functions import col, lower, regexp_replace, when, concat, lit

df_q12 = (
    df
    .withColumn(
        "username",
        lower(regexp_replace(col("Customer Name"), " ", ""))
    )
    .withColumn(
        "email",
        concat(
            lower(regexp_replace(col("Customer Name"), " ", "")),
            lit("@example.com")
        )
    )
)

df_q12.select("Customer Name", "email", "username").show(10, truncate=False)

+---------------+--------------------------+--------------+
|Customer Name  |email                     |username      |
+---------------+--------------------------+--------------+
|Claire Gute    |clairegute@example.com    |clairegute    |
|Claire Gute    |clairegute@example.com    |clairegute    |
|Darrin Van Huff|darrinvanhuff@example.com |darrinvanhuff |
|Sean O'Donnell |seano'donnell@example.com |seano'donnell |
|Sean O'Donnell |seano'donnell@example.com |seano'donnell |
|Brosina Hoffman|brosinahoffman@example.com|brosinahoffman|
|Brosina Hoffman|brosinahoffman@example.com|brosinahoffman|
|Brosina Hoffman|brosinahoffman@example.com|brosinahoffman|
|Brosina Hoffman|brosinahoffman@example.com|brosinahoffman|
|Brosina Hoffman|brosinahoffman@example.com|brosinahoffman|
+---------------+--------------------------+--------------+
only showing top 10 rows


In [12]:
df_q12_cleaned = df_q12.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

print("Records before cleaning:", df_q12.count())
print("Records after cleaning:", df_q12_cleaned.count())

df_q12_cleaned.select(
    "Customer Name",
    "email",
    "username"
).show(10, truncate=False)

Records before cleaning: 9994
Records after cleaning: 8662
+---------------+--------------------------+--------------+
|Customer Name  |email                     |username      |
+---------------+--------------------------+--------------+
|Claire Gute    |clairegute@example.com    |clairegute    |
|Claire Gute    |clairegute@example.com    |clairegute    |
|Darrin Van Huff|darrinvanhuff@example.com |darrinvanhuff |
|Sean O'Donnell |seano'donnell@example.com |seano'donnell |
|Sean O'Donnell |seano'donnell@example.com |seano'donnell |
|Brosina Hoffman|brosinahoffman@example.com|brosinahoffman|
|Brosina Hoffman|brosinahoffman@example.com|brosinahoffman|
|Brosina Hoffman|brosinahoffman@example.com|brosinahoffman|
|Brosina Hoffman|brosinahoffman@example.com|brosinahoffman|
|Brosina Hoffman|brosinahoffman@example.com|brosinahoffman|
+---------------+--------------------------+--------------+
only showing top 10 rows


In [13]:
# Q13: Prepare the price column for aggregation

from pyspark.sql.functions import expr

# The Superstore dataset does not have a 'price' column.
# Therefore, we use 'Sales' as the equivalent of price.
# try_cast() converts valid Sales values to double.
# Any invalid/non-numeric values are converted to NULL.

df_q13 = df.withColumn(
    "price",
    expr("try_cast(Sales as double)")
)

# Display Sales and the newly created price column
df_q13.select("Sales", "price").show(10)

+--------+--------+
|   Sales|   price|
+--------+--------+
|  261.96|  261.96|
|  731.94|  731.94|
|   14.62|   14.62|
|957.5775|957.5775|
|  22.368|  22.368|
|   48.86|   48.86|
|    7.28|    7.28|
| 907.152| 907.152|
|  18.504|  18.504|
|   114.9|   114.9|
+--------+--------+
only showing top 10 rows


In [14]:
# Q13: Calculate multiple statistics using .agg()

from pyspark.sql.functions import min, max, mean

# Use .agg() to calculate minimum, maximum,
# and mean price in a single operation

result_q13 = df_q13.agg(
    min("price").alias("Minimum_Price"),
    max("price").alias("Maximum_Price"),
    mean("price").alias("Mean_Price")
)

# Display the final aggregated result
result_q13.show()

+-------------+-------------+------------------+
|Minimum_Price|Maximum_Price|        Mean_Price|
+-------------+-------------+------------------+
|        0.444|     22638.48|234.41818199917006|
+-------------+-------------+------------------+



In [15]:
# Q15: Prepare the dataset for the final processing pipeline

from pyspark.sql.functions import col, expr

# Create 'store_id' using Customer ID
# and convert Sales into a numeric 'price' column

df_q15 = (
    df
    .withColumn("store_id", col("Customer ID"))
    .withColumn("price", expr("try_cast(Sales as double)"))
)

# Display prepared columns
df_q15.select(
    "store_id",
    "price"
).show(10)

+--------+--------+
|store_id|   price|
+--------+--------+
|CG-12520|  261.96|
|CG-12520|  731.94|
|DV-13045|   14.62|
|SO-20335|957.5775|
|SO-20335|  22.368|
|BH-11710|   48.86|
|BH-11710|    7.28|
|BH-11710| 907.152|
|BH-11710|  18.504|
|BH-11710|   114.9|
+--------+--------+
only showing top 10 rows


In [16]:
# Q15: Final data processing pipeline
from pyspark.sql.functions import sum
result_q15 = (
    df_q15
    # Step 1: Remove duplicate records
    .dropDuplicates()

    # Step 2: Replace null price values with 0
    .na.fill({"price": 0.0})

    # Step 3: Group records by store_id
    .groupBy("store_id")

    # Step 4: Calculate total revenue for each store
    .agg(
        sum("price").alias("Total_Revenue")
    )

    # Step 5: Sort by highest total revenue
    .orderBy(col("Total_Revenue").desc())
)
# Display final result
result_q15.show(20, truncate=False)

+--------+------------------+
|store_id|Total_Revenue     |
+--------+------------------+
|SM-20320|25043.05          |
|TC-20980|19017.847999999998|
|RB-19360|15117.338999999998|
|TA-21385|14595.62          |
|AB-10105|14355.610999999999|
|SC-20095|14142.333999999997|
|KL-16645|14071.916999999998|
|HL-15040|12873.297999999999|
|SE-20110|12209.438000000004|
|CC-12370|12129.072         |
|TS-21370|11885.871000000001|
|GT-14710|11820.119999999999|
|BM-11140|11609.9           |
|SV-20365|11420.645999999999|
|CJ-12010|11079.742         |
|CL-12565|10880.546         |
|ME-17320|10635.918000000001|
|KF-16285|10604.266         |
|BS-11365|10501.652999999998|
|EH-13765|10005.98          |
+--------+------------------+
only showing top 20 rows


In [17]:
# Save the final processed result as a CSV file

output_path = "/content/week5_final_result"

result_q15.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(output_path)

print("Final result saved successfully!")

Final result saved successfully!
